# tsconformal Real-Data Benchmark

This notebook executes the GPU-bound forecast-cache stage on Colab and packages the cached forecasts for transfer back to the local benchmark machine.

## Before running
1. Select `Runtime -> Change runtime type -> A100 GPU` (or `L4` / `T4`).
2. Create a Google Drive folder named `tsconformal_benchmark` if persistent storage is desired.
3. Upload these files to the Colab runtime:
   - `tsconformal.zip`
   - `electricity_hourly_dataset.zip`
   - `traffic_hourly_dataset.zip`
   - `fred_md_dataset.zip`

## Restart behavior
If Colab disconnects, reconnect and rerun the notebook. The cache step skips series that have already been written.


---
## Step 1: Configure persistent storage


In [ ]:
# Alternative: skip Drive, upload files directly
from google.colab import files
print("Upload tsconformal.zip")
files.upload()
print("Upload the 3 dataset zips")
files.upload()

In [ ]:
from pathlib import Path
import shutil
import subprocess

CONTENT_ROOT = Path('/content')
REPO_ARCHIVE = CONTENT_ROOT / 'tsconformal.zip'
REPO_ROOT = CONTENT_ROOT / 'tsconformal'

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = Path('/content/drive/MyDrive/tsconformal_benchmark')
    print('Google Drive mounted.')
except Exception as exc:
    DRIVE_ROOT = CONTENT_ROOT / 'tsconformal_output'
    print(f'Google Drive not available ({exc.__class__.__name__}: {exc}).')

CACHE_EXPORT_ROOT = DRIVE_ROOT / 'cached_forecasts'
RESULT_EXPORT_ROOT = DRIVE_ROOT / 'results'
for path in [CACHE_EXPORT_ROOT, RESULT_EXPORT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)


def copy_to_persistent_store(src, dst_dir):
    src = Path(src)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / src.name
    shutil.copy2(src, dst)
    print(f'Copied {src.name} -> {dst}')
    return dst


print(f'Persistent output root: {DRIVE_ROOT}')


---
## Step 2: Set up the repository and dependencies


In [ ]:
import os

os.chdir(CONTENT_ROOT)

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
macosx_dir = CONTENT_ROOT / '__MACOSX'
if macosx_dir.exists():
    shutil.rmtree(macosx_dir)

subprocess.run(['unzip', '-q', REPO_ARCHIVE.name], cwd=CONTENT_ROOT, check=True)
if macosx_dir.exists():
    shutil.rmtree(macosx_dir)

raw_dir = REPO_ROOT / 'data' / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)

for name in [
    'electricity_hourly_dataset.zip',
    'traffic_hourly_dataset.zip',
    'fred_md_dataset.zip',
]:
    src = CONTENT_ROOT / name
    if src.exists():
        shutil.move(str(src), raw_dir / name)
        print(f'Moved {name}')

print('\nDataset archives:')
for path in sorted(raw_dir.glob('*.zip')):
    print(f' - {path.name}')


In [ ]:
%%bash
cd /content/tsconformal
pip install -q -e ".[plots,dev]"
pip install -q ruptures statsmodels "chronos-forecasting>=2.0"

echo "---"
python -c "import tsconformal; print('tsconformal', tsconformal.__version__)"
python -c "import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"
python -c "from chronos import Chronos2Pipeline; print('Chronos2Pipeline: OK')"

---
## Step 3: Validate the data loaders


In [ ]:
%%bash
cd /content/tsconformal
PYTHONPATH=src:. python benchmarks/data_loaders.py

**Expected:** Electricity 321 series, Traffic 862 series, FRED-MD 107 series.

---
## Step 4: Smoke-test Chronos-2 on GPU


In [ ]:
import numpy as np
import torch
from chronos import Chronos2Pipeline

print(f"GPU: {torch.cuda.get_device_name(0)}")
pipeline = Chronos2Pipeline.from_pretrained('amazon/chronos-2', device_map='auto')
ctx = torch.tensor(np.random.randn(100).astype(np.float32))
q, _ = pipeline.predict_quantiles([ctx], prediction_length=1, quantile_levels=[0.1, 0.5, 0.9], batch_size=1)
print(f'Output: {q[0].detach().cpu().numpy().flatten()}')
print('✓ Chronos-2 OK')
del pipeline
torch.cuda.empty_cache()

---
## Step 5: Cache FM forecasts

The cache archives are written locally and then copied to the persistent output root when available.

If Colab disconnects, reconnect and rerun the notebook. Completed series are skipped.

A100 timing guide: FRED-MD about 10 min, Electricity about 30-60 min, Traffic about 1-3 hr.


In [ ]:
%%bash
cd /content/tsconformal
echo "===== FRED-MD ====="
PYTHONPATH=src:. python benchmarks/cache_fm_forecasts.py \
    --dataset fred_md --model chronos2 --device auto --series-batch-size 128
echo "Cached:"
find data/cached_forecasts/fred_md -name '*.json' 2>/dev/null | wc -l

In [ ]:
import tarfile
from google.colab import files

archive_path = CONTENT_ROOT / 'cached_fred_md.tar.gz'
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(
        REPO_ROOT / 'data' / 'cached_forecasts' / 'fred_md',
        arcname='data/cached_forecasts/fred_md',
    )
copy_to_persistent_store(archive_path, CACHE_EXPORT_ROOT)
files.download(str(archive_path))


In [ ]:
%%bash
cd /content/tsconformal
echo "===== ELECTRICITY ====="
PYTHONPATH=src:. python benchmarks/cache_fm_forecasts.py \
    --dataset electricity --model chronos2 --device auto --series-batch-size 64
echo "Cached:"
find data/cached_forecasts/electricity -name '*.json' 2>/dev/null | wc -l

In [ ]:
import tarfile
from google.colab import files

archive_path = CONTENT_ROOT / 'cached_electricity.tar.gz'
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(
        REPO_ROOT / 'data' / 'cached_forecasts' / 'electricity',
        arcname='data/cached_forecasts/electricity',
    )
copy_to_persistent_store(archive_path, CACHE_EXPORT_ROOT)
files.download(str(archive_path))


In [ ]:
%%bash
cd /content/tsconformal
echo "===== TRAFFIC ====="
PYTHONPATH=src:. python benchmarks/cache_fm_forecasts.py \
    --dataset traffic --model chronos2 --device auto --series-batch-size 32
echo "Cached:"
find data/cached_forecasts/traffic -name '*.json' 2>/dev/null | wc -l

In [ ]:
import tarfile
from google.colab import files

archive_path = CONTENT_ROOT / 'cached_traffic.tar.gz'
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(
        REPO_ROOT / 'data' / 'cached_forecasts' / 'traffic',
        arcname='data/cached_forecasts/traffic',
    )
copy_to_persistent_store(archive_path, CACHE_EXPORT_ROOT)
files.download(str(archive_path))


In [ ]:
# Cache summary
cache_root = REPO_ROOT / 'data' / 'cached_forecasts'
total = 0
for dataset_dir in sorted(cache_root.iterdir()) if cache_root.exists() else []:
    if dataset_dir.is_dir():
        for model_dir in sorted(dataset_dir.iterdir()):
            if model_dir.is_dir():
                n = len(list(model_dir.glob('*.json')))
                total += n
                print(f'{dataset_dir.name}/{model_dir.name}: {n} series')
print(f'\nTotal cached series: {total}')
print(f'Persistent cache export root: {CACHE_EXPORT_ROOT}')


---
## Step 6: Transfer the cached forecasts back to the local benchmark machine

The Colab stage ends here. The cached forecast archives created above should now be copied back to the local benchmark machine. The local machine then runs the real-data overlay, the synthetic benchmark, and the paper-artifact pipeline.

The corresponding local workflow is recorded in `reproducibility/colab_workflow.md`, `reproducibility/paper_alignment.md`, and `reproducibility/run_postcache_reproduction.sh`.
